In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import random
import numpy as np
from ast import literal_eval
from collections import Counter
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset with only necessary column
df = pd.read_csv("data/recipes.csv", usecols=["high_level_ingredients"])
df.dropna(inplace=True)

# Convert string to list
df["ingredients"] = df["high_level_ingredients"].apply(literal_eval)

# Flatten all ingredients for vocab building
all_ingredients = [ing for recipe in df["ingredients"] for ing in recipe]
ingredient_counts = Counter(all_ingredients)
min_freq = 5

# Build initial vocab with minimum frequency
filtered_ingredients = [ing for ing in all_ingredients if ingredient_counts[ing] >= min_freq]
unique_ingredients = sorted(set(filtered_ingredients))
vocab = {ing: idx for idx, ing in enumerate(unique_ingredients)}
inv_vocab = {idx: ing for ing, idx in vocab.items()}

# Filter ingredients in each recipe to those still in vocab
df["ingredients"] = df["ingredients"].apply(lambda lst: [ing for ing in lst if ing in vocab])
df = df[df["ingredients"].map(len) > 1]  # Remove recipes with less than 2 valid ingredients

# Generate skip-gram pairs (positive samples)
def generate_skipgram_pairs(recipes, window_size=2):
    pairs = []
    for recipe in recipes:
        idxs = [vocab[ing] for ing in recipe if ing in vocab]
        for i, center in enumerate(idxs):
            for j in range(max(i - window_size, 0), min(i + window_size + 1, len(idxs))):
                if i != j:
                    pairs.append((center, idxs[j]))
    return pairs

training_pairs = generate_skipgram_pairs(df["ingredients"].tolist())

# Define embedding model
class Ingredient2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, center, context):
        center_emb = self.embedding(center)
        context_emb = self.embedding(context)
        dot_product = torch.sum(center_emb * context_emb, dim=1)
        return dot_product

# Initialize model and optimizer
embedding_dim = 64
model = Ingredient2Vec(len(vocab), embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.BCEWithLogitsLoss()

# Generate training batches with negative sampling
def get_batches(pairs, batch_size=256):
    random.shuffle(pairs)
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        center, context = zip(*batch)
        center = torch.tensor(center, dtype=torch.long, device=device)
        context = torch.tensor(context, dtype=torch.long, device=device)
        labels = torch.ones(len(batch), dtype=torch.float, device=device)

        # Negative sampling
        neg_context = torch.randint(0, len(vocab), (len(batch),), device=device)
        neg_labels = torch.zeros(len(batch), dtype=torch.float, device=device)

        # Combine positive and negative samples
        combined_center = torch.cat([center, center])
        combined_context = torch.cat([context, neg_context])
        combined_labels = torch.cat([labels, neg_labels])

        yield combined_center, combined_context, combined_labels

# Train loop
for epoch in range(50):
    total_loss = 0
    for center_batch, context_batch, label_batch in get_batches(training_pairs):
        optimizer.zero_grad()
        outputs = model(center_batch, context_batch)
        loss = criterion(outputs, label_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/50], Loss: {loss.item():.4f}")

# Extract and normalize embeddings
embeddings = model.embedding.weight.detach().cpu().numpy()
embeddings = normalize(embeddings)

# Suggest substitutes
# def suggest_substitutes(ingredient, top_k=5):
#     if ingredient not in vocab:
#         return []
#     idx = vocab[ingredient]
#     vec = embeddings[idx]
#     sims = embeddings @ vec
#     top_indices = sims.argsort()[-top_k-1:-1][::-1]
#     return [inv_vocab[i] for i in top_indices]

def suggest_substitutes_cosine(ingredient, top_k=3):
    if ingredient not in vocab:
        return []
    idx = vocab[ingredient]
    vec = embeddings[idx].reshape(1, -1)
    sims = cosine_similarity(vec, embeddings)[0]
    top_idxs = sims.argsort()[-top_k-1:-1][::-1]
    return [inv_vocab[i] for i in top_idxs]
# Example usage
print(suggest_substitutes_cosine("egg"))

# Save embedding matrix and vocab mappings
with open("data/ingredient_embeddings.pkl", "wb") as f:
    pickle.dump({
        "embeddings": embeddings,  # normalized numpy array (vocab_size x embedding_dim)
        "vocab": vocab,            # dict: ingredient → index
        "inv_vocab": inv_vocab     # dict: index → ingredient
    }, f)
    
    
embeddings = normalize(model.embedding.weight.detach().cpu().numpy())


Using device: cpu
Epoch [10/50], Loss: 0.4841
Epoch [20/50], Loss: 0.4820
Epoch [30/50], Loss: 0.4078
Epoch [40/50], Loss: 0.4461
Epoch [50/50], Loss: 0.4725
['confectioners', 'milk', 'salt']


In [17]:
print(suggest_substitutes_cosine("cilantro"))


['cumin', 'hoisin sauce', 'fresh ginger']


In [13]:
import pandas as pd
from ast import literal_eval

df = pd.read_csv("data/recipes.csv", usecols=["high_level_ingredients", "diet_type"])

# Convert safely
df["high_level_ingredients"] = df["high_level_ingredients"].apply(literal_eval)

# Optional: simplify ingredient names (very basic cleanup)
def clean_ingredient_list(lst):
    return [i.lower().split()[-1] for i in lst if isinstance(i, str)]

df["cleaned_ingredients"] = df["high_level_ingredients"].apply(clean_ingredient_list)

print(df["cleaned_ingredients"].sample(5).tolist())


[['mustard', 'potatoes', 'weed', 'vinegar', 'cilantro', 'oil', 'onions', 'paprika', 'salt', 'celery', 'juice', 'pickles', 'pepper'], ['honey', 'pepper', 'oil', 'juice', 'nectar', 'tails', 'shrimp', 'garlic', 'sauce', 'ginger', 'sauce'], ['oil', 'salt', 'sugar', 'beans', 'pepper', 'cloves', 'seed', 'pepper'], ['cinnamon', 'milk', 'sugar', 'water'], ['powder', 'flour', 'sugar', 'salt', 'egg', 'milk', 'nutmeg', 'butter']]
